# Signal Lab

A compact numerical notebook using NumPy and SciPy to synthesize, filter, and inspect a signal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

rng = np.random.default_rng(42)
sample_rate = 400
seconds = 2.5
t = np.linspace(0, seconds, int(sample_rate * seconds), endpoint=False)
clean = np.sin(2 * np.pi * 8 * t) + 0.35 * np.sin(2 * np.pi * 42 * t)
observed = clean + 0.32 * rng.normal(size=t.size)
b, a = signal.butter(4, 18, btype='lowpass', fs=sample_rate)
filtered = signal.filtfilt(b, a, observed)

pd.DataFrame({
    'series': ['clean', 'observed', 'filtered'],
    'mean': [clean.mean(), observed.mean(), filtered.mean()],
    'std': [clean.std(), observed.std(), filtered.std()],
}).round(4)

## Time Domain

The filtered curve keeps the low-frequency structure and suppresses much of the noise.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(t, observed, color='#94a3b8', linewidth=0.8, label='observed')
ax.plot(t, filtered, color='#087f5b', linewidth=2, label='filtered')
ax.set_xlim(0, 1.2)
ax.set_xlabel('time (s)')
ax.set_ylabel('amplitude')
ax.set_title('Low-pass filtering')
ax.legend(frameon=False)
ax.grid(alpha=0.22)
fig.tight_layout()
plt.show()

## Frequency Domain

FFT and spectrogram outputs exercise heavier image rendering.

In [ ]:
freq = np.fft.rfftfreq(t.size, d=1 / sample_rate)
amp = np.abs(np.fft.rfft(observed)) / t.size
f, bins, spec = signal.spectrogram(observed, fs=sample_rate, nperseg=96, noverlap=64)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].plot(freq, amp, color='#1d4ed8')
axes[0].set_xlim(0, 80)
axes[0].set_title('FFT magnitude')
axes[0].set_xlabel('Hz')
axes[0].grid(alpha=0.2)
mesh = axes[1].pcolormesh(bins, f, spec, shading='auto', cmap='viridis')
axes[1].set_ylim(0, 90)
axes[1].set_title('Spectrogram')
axes[1].set_xlabel('time (s)')
axes[1].set_ylabel('Hz')
fig.colorbar(mesh, ax=axes[1], shrink=0.76)
fig.tight_layout()
plt.show()